---

# Step 7. 도전 과제 (실무 확장)

기본 실습이 끝났다면, 실무에서 챗봇을 만들 때 자주 마주치는 확장 작업을 시도해보세요. **코드 난이도를 높이는 게 아니라**, 실무에서 첫 번째 챗봇 빌드 후 바로 따라오는 작업들입니다.

## 🎯 과제 2: 페르소나별 temperature 다르게 설정 (실무 빈도 ★★★★☆)

1교시에서 배웠죠. `temperature`는 페르소나의 성격과도 연결됩니다.

- **분석가**: `temperature=0` (정확하고 일관된 답변)
- **디자이너**: `temperature=0.8` (창의적이고 다양한 답변)

**요구사항**
- `PERSONAS` 사전 구조를 `{name: {"system": ..., "temperature": ...}}` 형태로 확장
- `ask()` 함수에서 페르소나별 `temperature`로 호출
- **힌트**: `ChatOpenAI(model=..., temperature=...)`을 페르소나마다 새로 만들거나, `.bind(temperature=...)`을 사용





In [7]:
# 패키지 설치 (Colab/로컬 공통)
%pip install -q langchain langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")


✅ 로컬 Jupyter


In [9]:
# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


✅ API 키 OK


In [10]:
# 패키지 import
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

print("✅ import 완료")

✅ import 완료


In [11]:
# 페르소나 사전: 이름 → system 메시지
PERSONAS = {
    "분석가":  {"system": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.", "temperature": 0},
    "마케터":  {"system": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.", "temperature": 0.7},
    "개발자":  {"system": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.", "temperature": 0.5},
    "기획자":  {"system": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.", "temperature": 0.6},
    "디자이너": {"system": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.", "temperature": 0.8},
}


print(f"✅ {len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS.keys())}")

✅ 5개 페르소나 등록 완료: ['분석가', '마케터', '개발자', '기획자', '디자이너']


In [12]:
# 2교시에서 배운 기본 패턴
llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다."),
    ("human", "{question}"),
])

chain = prompt | llm   # LCEL 파이프 연산자 (2교시 3부)

# 동작 확인
result = chain.invoke({"role": "데이터 분석가", "question": "이상치란 무엇인가요? 한 문장으로.", "temperature": 0.3})
print(result.content)

이상치란 데이터 집합에서 다른 관측값들과 현저하게 다른 값을 의미합니다.


In [13]:
# 챗봇 핵심 함수: 페르소나 이름과 질문을 받아 답변 반환
def ask(persona_name: str, question: str) -> str:
    # 1. 페르소나 이름으로 system 메시지 조회
    persona = PERSONAS[persona_name]

    # 2. 체인 호출 (2교시에서 배운 prompt | llm 패턴)
    chain = prompt | llm.bind(temperature=persona["temperature"])  # 페르소나별 temperature 적용

    result = chain.invoke({"role": persona_name, "question": question})

    # 3. AIMessage 객체에서 답변 텍스트만 꺼내서 반환
    return result.content

In [14]:
# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

In [15]:
from datetime import datetime

# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    log = []   # 대화 누적 리스트

    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")


    while True:
        user_input = input("질문> ").strip()
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")
        
            log.append(f"[{datetime.now().strftime('%H:%M:%S')}] [{persona}] {user_input} → {answer}")

    # 종료 시:
    with open("chat_log.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(log))

In [16]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

[분석가] 안녕하세요! 어떻게 도와드릴까요?

[분석가] Python은 고급 프로그래밍 언어로, 1991년 귀도 반 로섬(Guido van Rossum)에 의해 처음 개발되었습니다. Python은 다음과 같은 특징을 가지고 있습니다:

1. **간결하고 읽기 쉬운 문법**: Python은 코드가 간결하고 명확하여, 초보자도 쉽게 배울 수 있습니다. 가독성이 높아 유지보수와 협업이 용이합니다.

2. **다양한 용도**: 웹 개발, 데이터 분석, 인공지능, 머신러닝, 자동화 스크립트, 게임 개발 등 다양한 분야에서 사용됩니다.

3. **풍부한 라이브러리**: Python은 다양한 라이브러리와 프레임워크를 제공하여, 개발자가 복잡한 작업을 쉽게 수행할 수 있도록 돕습니다. 예를 들어, NumPy, Pandas, Matplotlib, TensorFlow, Django 등이 있습니다.

4. **크로스 플랫폼**: Windows, macOS, Linux 등 다양한 운영체제에서 실행할 수 있습니다.

5. **대화형 프로그래밍**: Python은 대화형 모드에서 코드를 실행할 수 있어, 실험과 테스트가 용이합니다.

이러한 특성 덕분에 Python은 프로그래밍 언어 중에서 매우 인기 있는 선택이 되었습니다.

🎭 현재 페르소나: 분석가
📖 명령어: /quit, /exit, /change <이름>, /persona, /help
✅ 페르소나가 '개발자'(으)로 변경되었습니다.
[개발자] Python은 고급 프로그래밍 언어로, 1991년 귀도 반 로썸(Guido van Rossum)에 의해 처음 개발되었습니다. Python은 코드의 가독성이 뛰어나고, 간결한 문법을 가지고 있어 배우기 쉽고 사용하기 편리합니다. 이 언어는 다양한 프로그래밍 패러다임을 지원하며, 객체 지향, 절차적, 함수형 프로그래밍 등을 포함합니다.

Python의 주요 특징은 다음과 같습니다:

1. **가